In [ ]:
# only needs to be run once per session
%pip install -q xraydb beartype jaxtyping hdf5plugin h5py pyyaml

In [ ]:
# only needs to be run once per session
!apt-get install -y -q git-lfs
!git lfs install

In [ ]:
# !git clone https://github.com/noshou/APS360.git /kaggle/working/APS360  # first time only
!git -C /kaggle/working/APS360 pull                                        # run to get latest code

In [ ]:
# ── rclone / Google Drive setup ── run once per session ─────────────────────
# Mirrors checkpoints off-box so a session timeout doesn't lose them.
# Prereq (one-time): configure rclone locally, then paste the contents of your
# ~/.config/rclone/rclone.conf into a Kaggle Secret named RCLONE_CONF
# (Add-ons -> Secrets). Set ckpt_rclone_dest in the config cell to your remote.
!curl -fsSL https://rclone.org/install.sh | sudo bash

import os
from kaggle_secrets import UserSecretsClient
os.makedirs("/root/.config/rclone", exist_ok=True)
with open("/root/.config/rclone/rclone.conf", "w") as f:
    f.write(UserSecretsClient().get_secret("RCLONE_CONF"))

!rclone listremotes          # sanity check: should print your remote, e.g. "gdrive:"


In [ ]:
# ── Verbosity ─────────────────────────────────────────────────────────────────
#
#   "epoch"      — one summary line per epoch (train/val/test loss + R²)
#   "batch"      — also prints running-average loss every 20 batches
#   "diagnostic" — per-batch NaN/Inf check with full tensor stats on the first
#                  10 batches; use when debugging numerical issues
#
VERBOSITY = "batch"  

import sys, os
sys.path.insert(0, "/kaggle/working/APS360")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True" # prevents fragmentation
os.environ["PYTHONPATH"] = "/kaggle/working/APS360"           # inherited by mp.spawn workers

# clear python cache
for mod in list(sys.modules.keys()):
    if "ScatterNet" in mod or "train" in mod:
        del sys.modules[mod]

from ScatterNet.config import RunConfig, DEFAULT_BUCKETS
from train import main

# bin 58 contains molecules up to 78,819 atoms; even with 2-GPU TP the shard
# (~39k atoms) exhausts T4 memory. Drop it — max molecule becomes 6,046 atoms.
SAFE_BUCKETS = [b for b in DEFAULT_BUCKETS if b[1] <= 6046]

cfg = RunConfig(

    # --- paths ---
    hdf5           = "/kaggle/input/datasets/noso0s0n/iql50/I(q)L50.h5", # HDF5 dataset
    db             = "/kaggle/working/APS360/Preprocess/scatternet",     # SQLite encoding stem
    ckpt_best      = "/kaggle/working/scatternet_best.pt",               # LOCAL staging: best-val model (auto-copied to ckpt_rclone_dest)
    ckpt_resume    = "/kaggle/working/scatternet_resume.pt",             # LOCAL staging: latest state for crash-resume (copied to Drive every ckpt_interval_sec + each epoch)
    metrics        = "/kaggle/working/scatternet_metrics.json",          # LOCAL staging: per-epoch loss/R2 log (auto-copied to ckpt_rclone_dest)
    resume         = None,                                               # path to resume from, or None

    # --- checkpointing / crash-safety ---
    ckpt_rclone_dest  = "your/remote/here", # rclone dest; checkpoints copied here after every save
    ckpt_interval_sec = 600,                # save a mid-epoch resume point every 10 min

    # --- model ---
    lambda_1       = 128,  # atom embedding dimension
    lambda_2       = 5,    # message passing rounds
    lambda_3       = 128,  # OutputHead hidden width
    lambda_4       = 4,    # MLP halving steps (2^lambda_4 <= lambda_3)
    lambda_5       = 128,  # Random Fourier Features
    msg_seed       = 42,   # RFF frequency matrix seed
    atm_chunk      = 64,   # atoms per M-chunk
    mol_chunk      = 64,   # molecules per N-chunk
    eps_embd       = 1e-8, # numerical floor in Embed
    eps_msgp       = 1e-3, # floor in MessagePass (Å)

    # --- loss ---
    lambda_6       = 0.1,   # form-factor penalty weight
    lambda_7       = 0.1,   # sigma inverse-L1 regularisation weight
    eps_sigma      = 1e-4,  # floor added to sigma before inverse-L1 penalty (prevents 1/sigma -> inf)

    # --- training ---
    lr             = 3e-4,  # Adam learning rate
    weight_decay   = 1e-5,  # Adam L2 weight decay
    grad_clip      = 1.0,   # max gradient norm
    epochs         = 50,    # epochs to train
    batcher_seed   = 0,     # train/val/test split seed
    atom_size_ceil = 78819, # max atoms per batch — MUST be large; small values create too many batches
    num_workers    = 3,     # DataLoader workers
    max_batches    = None,  # cap batches per epoch (None = no limit)
    verbosity      = VERBOSITY,

    # --- data ---
    buckets        = SAFE_BUCKETS,
)


In [ ]:
# Fresh training run (resume = None).
main(cfg)


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# RESUME after a crash / session timeout
# ═══════════════════════════════════════════════════════════════════════════
# Training saves a resume checkpoint to Drive every ckpt_interval_sec. To resume:
#   1. Re-run the cells above EXCEPT the "Fresh training run" cell:
#        deps -> git-lfs -> git pull -> rclone setup -> the RunConfig cell
#        (the RunConfig cell defines `cfg` and imports `main`; it no longer
#         starts training, so running it is safe).
#   2. Run THIS cell. It pulls the latest checkpoint from Drive and continues
#      from the saved batch. Mid-epoch resume is exact (the per-epoch seed makes
#      the shuffle reproducible), so you lose at most ckpt_interval_sec of work.
import dataclasses

# pull the last checkpoint back from Drive (matches ckpt_rclone_dest + basename of ckpt_resume)
!rclone copy "your/remote/here/scatternet_resume.pt" /kaggle/working/

resume_cfg = dataclasses.replace(cfg, resume="/kaggle/working/scatternet_resume.pt")
main(resume_cfg)
